# Notebook pour charger le jeu sur VS Code

In [1]:
import gymnasium as gym
import ale_py

print(f"Version de ALE: {ale_py.__version__}")

game_list = [spec for spec in gym.registry.keys() if 'IceHockey' in spec]
print(f"Jeux d'arcade disponibles: {game_list}")

Version de ALE: 0.11.2
Jeux d'arcade disponibles: ['IceHockey-v0', 'IceHockey-v4', 'IceHockeyNoFrameskip-v0', 'IceHockeyNoFrameskip-v4', 'ALE/IceHockey-v5']


In [2]:
from pettingzoo.atari import ice_hockey_v2
import pygame

try:
    env = ice_hockey_v2.env(render_mode="human")
    env.reset()
    print("Succès ! Le jeu est prêt.")
    env.close()
except Exception as e:
    print(f"Erreur persistante : {e}")

Succès ! Le jeu est prêt.


In [3]:
from pettingzoo.atari import ice_hockey_v2

env = ice_hockey_v2.env(render_mode="human")
env.reset()

# On définit une limite de 5000 itérations
max_steps = 5000

for i, agent in enumerate(env.agent_iter()):
    observation, reward, termination, truncation, info = env.last()

    if termination or truncation :
        action = None
    else:
        action = env.action_space(agent).sample()

    env.step(action)
    
    # Sortir de la boucle si on a atteint la limite ou la fin du jeu
    if i >= max_steps:
        print(f"Arrêt manuel après {max_steps} itérations.")
        break

env.close()
print("Démo terminée proprement !")

Arrêt manuel après 5000 itérations.
Démo terminée proprement !


# Exécution de MA-POCA

In [5]:
# importation des bibliothèques et des classes nécessaires

from MAPOCA.multi_agent_buffer import MultiAgentBuffer

buffer = MultiAgentBuffer()


In [ ]:
import torch

def collect_trajectories(env, actor, buffer, max_steps=1000):

    "Cette fonction permet de récupérer les trajectoires des agents pendant les mille premières étapes du jeu"
    env.reset()

    # Initialisation des stockages temporaires par agent
    temp_obs = {}
    temp_acts = {}
    temp_logprobs = {}

    step_count = 0

    for agent in env.agent_iter():
        # 'info' est ignoré car non nécessaire pour la logique MA-POCA de base
        obs, reward, terminated, truncated, _ = env.last()
        done = terminated or truncated

        if done:
            action = None
            logp = 0
        else:
            # Conversion de l'observation (H, W, C) -> (1, C, H, W) pour le CNN
            # Note: Vérifie si ton Wrapper Supersuit a déjà mis les channels en premier
            obs_tensor = torch.from_numpy(obs).float().unsqueeze(0)

            with torch.no_grad():
                logits = actor(obs_tensor)
                dist = torch.distributions.Categorical(logits=logits)
                action_tensor = dist.sample()
                logp = dist.log_prob(action_tensor).item()
                action = action_tensor.item()

        env.step(action)

        # 1. Stockage de l'observation de l'agent actuel
        temp_obs[agent] = obs
        
        # 2. Stockage de l'action si le jeu n'est pas fini
        if action is not None:
            temp_acts[agent] = action
            temp_logprobs[agent] = logp
        
        # 3. Une fois que les DEUX agents ont agi (Fin du cycle temporel t)
        if len(temp_acts) == 2:
            # On récupère le dictionnaire complet des récompenses du pas t
            # 'env.rewards' contient {'first_0': r1, 'second_0': r2}
            current_rewards = [env.rewards['first_0'], env.rewards['second_0']]

            buffer.add(
                observation=[temp_obs['first_0'], temp_obs['second_0']],
                action=[temp_acts['first_0'], temp_acts['second_0']],
                logprob=[temp_logprobs['first_0'], temp_logprobs['second_0']],
                reward=current_rewards,
                done=done
            )

            # Reset des dictionnaires temporaires pour le prochain pas t+1
            temp_acts = {}
            temp_logprobs = {}
            step_count += 1

        if done or step_count >= max_steps:
            break
    
    return step_count